In [1]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling, AutoModelForSeq2SeqLM, AutoModelForMaskedLM, AutoModelForCausalLM
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch
import os


os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

# 1. 加载模型和分词器
# model_name = "EleutherAI/gpt-neo-125M"
# model_name = "facebook/opt-125m"
# model_name = "cerebras/Cerebras-GPT-111M"
# model_name = "bigscience/bloom-560m" # I should try it on CoLab
# model_name = "mosaicml/mpt-7b"
# model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

# model_name = "t5-base"
# model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


model_name = "microsoft/deberta-base"
model = AutoModelForMaskedLM.from_pretrained(model_name)


tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


# 2. 加载数据集
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# 3. 分词
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True, remove_columns=["text"])

# 4. 创建 DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# 5. 创建 DataLoader
tokenized_datasets.set_format("torch")
dataloader = DataLoader(tokenized_datasets, batch_size=10, shuffle=True, collate_fn=data_collator)

# 6. 设置优化器
optimizer = AdamW(model.parameters(), lr=5e-5)

# 7. 训练循环
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
device = torch.device("cpu")
model.to(device)
model.train()

from experiments.trainer.plugins import SnapshotPlugin, ProfilerPlugin
from perf_estimator.config import Config
snap_conf = Config(save2tmp=False)
snapshot = SnapshotPlugin(config=snap_conf)
profiler = ProfilerPlugin(config=snap_conf)

epochs = 1
snapshot.start()
profiler.start()
for epoch in range(epochs):
    for index, batch in enumerate(dataloader):
        snapshot.step()
        profiler.step()
        with torch.set_grad_enabled(True):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            print(f"Epoch {epoch}, Loss: {loss.item()}")
            if index == 3:
                break
snapshot.stop()
profiler.stop()

print("Train Finsihed！")

/tmp/ipykernel_190037/3592799379.py:78: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/aten/src/ATen/native/Scalar.cpp:22.)
  print(f"Epoch {epoch}, Loss: {loss.item()}")


Epoch 0, Loss: 4.286785125732422


[W419 00:58:50.327106320 CPUAllocator.cpp:245] Memory block of unknown size was allocated before the profiling started, profiler results will not include the deallocation event


Epoch 0, Loss: 4.201906681060791
Epoch 0, Loss: 3.923902750015259
Epoch 0, Loss: 3.95957350730896
Train Finsihed！


In [2]:
d = next(iter(dataloader))

In [3]:
getattr(d, 'data')

{'input_ids': tensor([[    2,  3515,    83,  ...,     2,     2,     2],
         [    2,   598, 20679,  ...,     2,     2,     2],
         [    2,    20, 25501,  ...,     2,     2,     2],
         ...,
         [    2, 13550, 18195,  ...,     2,     2,     2],
         [    2,  2240,  4569,  ...,    54,  1550,     5],
         [    2,     2,     2,  ...,     2,     2,     2]]),
 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 0, 0, 0],
         ...,
         [1, 1, 1,  ..., 0, 0, 0],
         [1, 1, 1,  ..., 1, 1, 1],
         [1, 0, 0,  ..., 0, 0, 0]]),
 'labels': tensor([[ -100,  3515,    83,  ...,  -100,  -100,  -100],
         [ -100,   598, 20679,  ...,  -100,  -100,  -100],
         [ -100,    20, 25501,  ...,  -100,  -100,  -100],
         ...,
         [ -100, 13550, 18195,  ...,  -100,  -100,  -100],
         [ -100,  2240,  4569,  ...,    54,  1550,     5],
         [ -100,  -100,  -100,  ...,  -100,  -100,  -1